# SMIC

Code for *Benchmark Retrievability and Symmetry-Induced Memory Collapse in Sign Language Recognition*.

Open this notebook in Colab: [colab.research.google.com/github/sachitrazz/SMIC](https://colab.research.google.com/github/sachitrazz/SMIC/blob/main/SMIC.ipynb)

**The corpora are not redistributed.** ISL-IEEE and ASL-IEEE are on IEEE DataPort, and CSL comes from its authors. So this notebook runs the parts that need no data:

1. the smoke tests,
2. the numerical verification of the three Appendix propositions,
3. regeneration of the paper's tables and figures from the committed `results/*.json`.

The last section shows how to point the code at your own copy of the corpora.

## 1. Get the code

In [ ]:
import os

if not os.path.isdir('SMIC'):
    !git clone --quiet https://github.com/sachitrazz/SMIC.git
os.chdir('/content/SMIC' if os.path.isdir('/content/SMIC') else 'SMIC')
print('working directory:', os.getcwd())
print('files:', len(os.listdir('.')))

Colab already provides PyTorch, NumPy, SciPy and matplotlib, which is all the
no-data sections need. The hand-crop preprocessing additionally needs
`mediapipe` and `opencv-python`; install those only if you are preparing data.

In [ ]:
import torch, numpy, scipy, matplotlib

print('torch      ', torch.__version__)
print('numpy      ', numpy.__version__)
print('scipy      ', scipy.__version__)
print('matplotlib ', matplotlib.__version__)
print('GPU        ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only, fine for this notebook)')

## 2. Smoke tests

Eleven tests, no dataset required: the Appendix propositions, the referenced
STLAT model building and running, seed reproducibility, and configuration.

In [ ]:
!python -m unittest discover -s tests -v

## 3. The three propositions, verified numerically

Every quantitative claim in the paper's Appendix is checked by brute force over
randomised instances.

In [ ]:
!python theory.py

## 4. Rebuild the paper's tables and figures

Every number in the paper is stored in `results/*.json`, which is committed, so
the tables and figures regenerate without the corpora.

In [ ]:
!python tools/make_bench_table.py
!python tools/make_dataset_table.py
!python tools/make_leakage_tables.py

In [ ]:
!cd figures && python make_bench_figures.py && python make_theory_figure.py && python make_leakage_figure.py

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('figures/*.jpg')):
    print(path)
    display(Image(filename=path, width=900))

## 5. Running with your own copy of the corpora

Obtain ISL-IEEE (doi:10.21227/796w-a432) and ASL-IEEE (doi:10.21227/4dz0-xv55)
from IEEE DataPort, and CSL from its authors. Mount your Drive, point the
environment variables at the folders, then follow the run order in the README.

```python
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['ISL_ROOT']  = '/content/drive/MyDrive/ISL_DATA_IEEE_TRAIN'
os.environ['CSL_TRAIN'] = '/content/drive/MyDrive/CSL_Train'
os.environ['CSL_TEST']  = '/content/drive/MyDrive/CSL_Test'
```

Then, in order:

```
!pip install -q mediapipe opencv-python
!python prepare_new.py          # deduplication and hand crops
!python regroup_split.py        # near-duplicate-group-disjoint splits
!python prepare_csl_signer.py   # CSL sign frames, signer-disjoint task
!python run_all.py audit        # the split-integrity audit
!python bench.py isl asl csl    # the benchmark (GPU recommended)
```

Set `SMIC_DETERMINISTIC=1` if you want bit-identical GPU runs. It is off by
default, because the paper's numbers were produced without it.